In [2]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder
import pickle
data=pd.read_csv("Churn_Modelling.csv")
data.head()

data=data.drop(['RowNumber','CustomerId','Surname'],axis=1)

label_encoder_gender=LabelEncoder()
data['Gender']=label_encoder_gender.fit_transform(data['Gender'])

from sklearn.preprocessing import OneHotEncoder

onehot_encoder_geo=OneHotEncoder()

geo_encoder=onehot_encoder_geo.fit_transform(data[['Geography']]).toarray()
onehot_encoder_geo.get_feature_names_out(['Geography'])

geo_encoded_df=pd.DataFrame(
    geo_encoder,
    columns=onehot_encoder_geo.get_feature_names_out(['Geography'])
)

data=pd.concat(
    [data.drop('Geography',axis=1),geo_encoded_df],
    axis=1
)

with open('label_encoder_gender.pkl','wb') as file:
    pickle.dump(label_encoder_gender,file)
with open('onehot_encoder_geo.pkl','wb') as file:
    pickle.dump(onehot_encoder_geo,file)

X=data.drop('Exited',axis=1)
y=data['Exited']  

X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2,
    random_state=42
)

scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

with open('scaler.pkl','wb') as file:
    pickle.dump(scaler,file)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime

model=Sequential([
    Dense(64, activation='relu',input_shape=(X_train.shape[1],)),
    Dense(32,activation='relu'),
    Dense(1,activation='sigmoid')

])
model.summary()

opt=tf.keras.optimizers.Adam(learning_rate=0.01)
loss=tf.keras.losses.BinaryCrossentropy()


model.compile(
    optimizer=opt,
    loss="binary_crossentropy",
    metrics=['accuracy']
)

log_dir="logs/fit/"+datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

tensorboard_callback=TensorBoard(
    log_dir=log_dir,
    histogram_freq=1
)

early_stopping_callback=EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)
history=model.fit(
    X_train,
    y_train,
    validation_data=(X_test,y_test),
    epochs=100,
    callbacks=[tensorboard_callback,early_stopping_callback]
)

model.save('model.h5')
%load_ext tensorboard
%tensorboard --logdir logs/fit --port 6008

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_3 (Dense)             (None, 64)                832       
                                                                 
 dense_4 (Dense)             (None, 32)                2080      
                                                                 
 dense_5 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
Epoch 1/100
250/250 [==============================] - 2s 5ms/step - loss: 0.3937 - accuracy: 0.8334 - val_loss: 0.3626 - val_accuracy: 0.8515
Epoch 2/100
250/250 [==============================] - 1s 4ms/step - loss: 0.3553 - accuracy: 0.8546 - val_loss: 0.3504 - val_accuracy: 0.857

D:\Anaconda\envs\V-ANN\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


Reusing TensorBoard on port 6008 (pid 8220), started 3 days, 10:24:59 ago. (Use '!kill 8220' to kill it.)

In [3]:
import tensorflow as tf
import pickle
import pandas as pd

model = tf.keras.models.load_model('model.h5')

with open('label_encoder_gender.pkl','rb') as file:
    label_encoder_gender = pickle.load(file)

with open('onehot_encoder_geo.pkl','rb') as file:
    label_encoder_geo = pickle.load(file)

with open('scaler.pkl','rb') as file:
    scaler = pickle.load(file)

input_data = {
    'CreditScore': 500,
    'Geography': 'Germany',
    'Gender': 'Male',
    'Age': 55,
    'Tenure': 5,
    'Balance': 50000,
    'NumOfProducts': 4,
    'HasCrCard': 0,
    'IsActiveMember': 1,
    'EstimatedSalary': 40000
}

geo_encoded = label_encoder_geo.transform([[input_data['Geography']]]).toarray()

geo_encoded_df = pd.DataFrame(
    geo_encoded,
    columns=label_encoder_geo.get_feature_names_out(['Geography'])
)

input_df = pd.DataFrame([input_data])

input_df['Gender'] = label_encoder_gender.transform(input_df['Gender'])

input_df = pd.concat(
    [input_df.drop('Geography', axis=1), geo_encoded_df],
    axis=1
)

input_scaled = scaler.transform(input_df)

prediction = model.predict(input_scaled)
prediction_proba = prediction[0][0]

if prediction_proba > 0.5:
    print('The customer is likely to churn.')
else:
    print('The customer is not likely to churn.')

D:\Anaconda\envs\V-ANN\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


1/1 [==============================] - 0s 264ms/step
The customer is likely to churn.


In [4]:
%%writefile Tapp.py
import streamlit as st
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
import pandas as pd
import pickle


model = tf.keras.models.load_model('model.h5')

with open('label_encoder_gender.pkl','rb') as file:
    label_encoder_gender = pickle.load(file)

with open('onehot_encoder_geo.pkl','rb') as file:
    onehot_encoder_geo = pickle.load(file)

with open('scaler.pkl','rb') as file:
    scaler = pickle.load(file)

st.title("Customer Churn Prediction")

geography = st.selectbox('Geography', onehot_encoder_geo.categories_[0])
gender = st.selectbox('Gender', label_encoder_gender.classes_)
age = st.slider('Age',18,92)
balance = st.number_input('Balance')
credit_score = st.number_input('Credit Score')
estimated_salary = st.number_input('Estimated Salary')
tenure = st.slider('Tenure',0,10)
num_of_products = st.slider('Number of Products', 1,4)
has_cr_card = st.selectbox('Has Credit Card',[0,1])
is_active_member = st.selectbox('Is Active Member',[0,1])

input_data = pd.DataFrame({
    'CreditScore': [credit_score],
    'Gender': [label_encoder_gender.transform([gender])[0]],
    'Age': [age],
    'Tenure': [tenure],
    'Balance': [balance],
    'NumOfProducts': [num_of_products],
    'HasCrCard': [has_cr_card],
    'IsActiveMember': [is_active_member],
    'EstimatedSalary': [estimated_salary]
})

geo_encoded = onehot_encoder_geo.transform([[geography]]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=onehot_encoder_geo.get_feature_names_out(['Geography']))

input_data = pd.concat([input_data.reset_index(drop=True),geo_encoded_df], axis=1)

input_data_scaled = scaler.transform(input_data)

prediction = model.predict(input_data_scaled)
prediction_proba = prediction[0][0]

st.write(f'Churn Probability: {prediction_proba:.2f}')

if prediction_proba > 0.5 :
    st.write('The Customer is likely to churn')
else:
    st.write('The customer is not likely to churn')

Writing Tapp.py
